In [1]:
!pip install -q google-generativeai datasets pandas tqdm langsmith evaluate nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00


In [2]:
!pip uninstall -y evaluate datasets --q

In [ ]:
!pip install evaluate
!pip install rouge_score

In [ ]:
# https://www.langchain.com/langsmith/observability
# profile > api_keys > create api key and store the key

In [4]:
%%writefile .env
GENAI_API_KEY=
LANGSMITH_API_KEY=

Writing .env


In [5]:
from dotenv import load_dotenv #load the api keys we have enviorment file [gemini and langsmith]
import os #list dir, mkdir, etc
import google.generativeai as genai  #gemini llms
from langsmith import Client # observability

load_dotenv() # load all the api keys from .env

genai.configure(api_key=os.getenv("GENAI_API_KEY")) # configure gemini
client = Client(api_key=os.getenv("LANGSMITH_API_KEY")) #configure langsmith

In [6]:
dataset = [
    {
        "question": "What is the notice period for termination?",
        "contexts": [
            "The agreement may be terminated by either party with 30 days written notice."
        ],
        "ground_truth": "30 days written notice"
    },
    {
        "question": "Is the agreement governed by US law?",
        "contexts": [
            "This agreement shall be governed by and construed in accordance with the laws of the State of California."
        ],
        "ground_truth": "Yes, governed by California (US) law"
    },
    {
        "question": "Who bears liability for software malfunction?",
        "contexts": [
            "The vendor shall not be liable for indirect or consequential damages caused by software malfunction."
        ],
        "ground_truth": "The vendor is not liable"
    },
    {
        "question": "Can the contract be renewed automatically?",
        "contexts": [
            "The contract will automatically renew for successive one-year terms unless terminated in writing."
        ],
        "ground_truth": "Yes, auto-renews unless terminated"
    },
    {
        "question": "What is the payment due date?",
        "contexts": [
            "Payment is due within 15 days from the invoice date."
        ],
        "ground_truth": "15 days from invoice date"
    }
]

print("Loaded custom QA dataset. Example:")
print(dataset[0])

Loaded custom QA dataset. Example:
{'question': 'What is the notice period for termination?', 'contexts': ['The agreement may be terminated by either party with 30 days written notice.'], 'ground_truth': '30 days written notice'}


In [16]:
import uuid, time
from tqdm import tqdm

model = genai.GenerativeModel("gemini-2.0-flash-lite")

def generate_answer(question, contexts):
    prompt = f"""You are a legal AI assistant. Use the following context to answer the question.

Context:
{chr(10).join(contexts[:3])}

Question: {question}
Answer:"""
    response = model.generate_content(prompt)
    return response.text.strip()

def log_to_langsmith(question, contexts, answer):
    run_id = str(uuid.uuid4())
    client.create_run(
        name="Gemini-LegalQA_class",
        run_type="llm",
        inputs={"question": question, "contexts": contexts},
        outputs={"response": answer},
        tags=["legal", "gemini", "eval"],
        run_id=run_id,
        start_time=time.time(),
        end_time=time.time()
    )
    return run_id

In [17]:
# This cell is from "Generate Responses (10 Samples)"
generated = []
print("Generating answers with logging...")

for i, sample in enumerate(tqdm(dataset)):
    print(f"Processing sample {i+1}/{len(dataset)}...")
    result = generate_and_log(sample["question"], sample["contexts"])
    generated.append(result)
    print(f"Finished sample {i+1}")
    import time
    time.sleep(1) # Add a small delay


Generating answers with logging...


  0%|          | 0/5 [00:00<?, ?it/s]

Processing sample 1/5...
Finished sample 1


 20%|██        | 1/5 [00:02<00:08,  2.23s/it]

Processing sample 2/5...
Finished sample 2


...

 60%|██████    | 3/5 [00:06<00:04,  2.30s/it]

Processing sample 4/5...
Finished sample 4


 80%|████████  | 4/5 [00:08<00:02,  2.20s/it]

Processing sample 5/5...
Finished sample 5


100%|██████████| 5/5 [00:11<00:00,  2.21s/it]


In [18]:
import pandas as pd

df = pd.DataFrame(dataset)
df["generated_answer"] = generated

print("DataFrame created with generated answers.\n")
print(df[["question", "generated_answer"]].head())

print("Example Generated Output:\n")
print(df["generated_answer"][0])

DataFrame created with generated answers.

                                        question  \
0     What is the notice period for termination?   
1           Is the agreement governed by US law?   
2  Who bears liability for software malfunction?   
3     Can the contract be renewed automatically?   
4                  What is the payment due date?   

                                    generated_answer  
0  The notice period for termination is 30 days w...  
1  The agreement is governed by the laws of the S...  
2  The vendor does not bear liability for indirec...  
3    Yes, the contract can be renewed automatically.  
4  The payment is due within 15 days from the inv...  
Example Generated Output:

The notice period for termination is 30 days written notice.


In [19]:
df.head()

,question,contexts,ground_truth,generated_answer
0,What is the notice period for termination?,[The agreement may be terminated by either par...,30 days written notice,The notice period for termination is 30 days w...
1,Is the agreement governed by US law?,[This agreement shall be governed by and const...,"Yes, governed by California (US) law",The agreement is governed by the laws of the S...
2,Who bears liability for software malfunction?,[The vendor shall not be liable for indirect o...,The vendor is not liable,The vendor does not bear liability for indirec...
3,Can the contract be renewed automatically?,[The contract will automatically renew for suc...,"Yes, auto-renews unless terminated","Yes, the contract can be renewed automatically."
4,What is the payment due date?,[Payment is due within 15 days from the invoic...,15 days from invoice date,The payment is due within 15 days from the inv...


In [20]:
from evaluate import load

# Try with force_download=True
bleu = load("bleu", force_download=True)
# Or, if that doesn't work, ensure local_files_only is False (default)
# bleu = load("bleu", local_files_only=False)

bleu_scores = []
for i in range(len(df)):
    result = bleu.compute(predictions=[df["generated_answer"][i]], references=[df["ground_truth"][i]])
    bleu_scores.append(result["bleu"])

df["bleu"] = bleu_scores

print("BLEU Score (avg):", sum(bleu_scores)/len(bleu_scores))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BLEU Score (avg): 0.046924700641056


In [23]:
from evaluate import load

# Load ROUGE metric
rouge = load("rouge", force_download=True)

rouge_scores = {
    "rouge1": [],
    "rouge2": [],
    "rougeL": [],
    "rougeLsum": []
}

for i in range(len(df)):
    result = rouge.compute(
        predictions=[df["generated_answer"][i]],
        references=[df["ground_truth"][i]]
    )

    rouge_scores["rouge1"].append(result["rouge1"])
    rouge_scores["rouge2"].append(result["rouge2"])
    rouge_scores["rougeL"].append(result["rougeL"])
    rouge_scores["rougeLsum"].append(result["rougeLsum"])

# Add to DataFrame
df["rouge1"] = rouge_scores["rouge1"]
df["rouge2"] = rouge_scores["rouge2"]
df["rougeL"] = rouge_scores["rougeL"]
df["rougeLsum"] = rouge_scores["rougeLsum"]

# Print average scores
print("ROUGE-1 (avg):", sum(df["rouge1"])/len(df))
print("ROUGE-2 (avg):", sum(df["rouge2"])/len(df))
print("ROUGE-L (avg):", sum(df["rougeL"])/len(df))
print("ROUGE-Lsum (avg):", sum(df["rougeLsum"])/len(df))


ROUGE-1 (avg): 0.36246979388770434
ROUGE-2 (avg): 0.22024420024420022
ROUGE-L (avg): 0.36246979388770434
ROUGE-Lsum (avg): 0.36246979388770434


In [24]:
#LLM TO REFER TO CONTEXT AND ANSWER TO PROVIDE A SCORE
def score_faithfulness_llm(contexts, answer):
    context_text = "\n".join(contexts)

    prompt = f"""
You are an evaluator. Determine if the answer is fully supported by the provided context.

CONTEXT:
{context_text}

ANSWER:
{answer}

Return:
1 = The answer is fully grounded in the context (faithful)
0 = The answer includes information NOT supported by the context (hallucination)

Respond with only 0 or 1.
"""

    result = model.generate_content(prompt)
    score = result.text.strip()

    return int(score) if score in ["0", "1"] else 0


In [25]:
# LLM TO REFER TO GROUND TRUTH AND ANSWER AND PROVIDE A SCORE
def score_factuality_llm(ground_truth, answer):
    prompt = f"""
You are an evaluator. Compare the answer with the reference ground truth.

GROUND TRUTH:
{ground_truth}

ANSWER:
{answer}

TASK:
Score factual correctness.

Return:
1 = The answer is factually correct and aligns with the ground truth
0 = The answer is incorrect or contradicts the ground truth

Respond with only 0 or 1.
"""

    result = model.generate_content(prompt)
    score = result.text.strip()

    return int(score) if score in ["0", "1"] else 0


In [ ]:
# LLM TO REFER TO question AND ANSWER AND PROVIDE A SCORE
def score_relevance_llm(question, answer):
    prompt = f"""
You are an evaluator. Compare the answer with the reference ground truth.

QUESTION:
{question}

ANSWER:
{answer}

TASK:
Score relevance.

Return:
1 = The answer is factually correct and aligns with the question
0 = The answer is incorrect or contradicts the question

Respond with only 0 or 1.
"""

    result = model.generate_content(prompt)
    score = result.text.strip()

    return int(score) if score in ["0", "1"] else 0


In [26]:
df["faithfulness_llm"] = df.apply(
    lambda row: score_faithfulness_llm(row["contexts"], row["generated_answer"]),
    axis=1
)

df["factuality_llm"] = df.apply(
    lambda row: score_factuality_llm(row["ground_truth"], row["generated_answer"]),
    axis=1
)


In [27]:
df

,question,contexts,ground_truth,generated_answer,bleu,rouge1,rouge2,rougeL,rougeLsum,faithfulness_llm,factuality_llm
0,What is the notice period for termination?,[The agreement may be terminated by either par...,30 days written notice,The notice period for termination is 30 days w...,0.234624,0.571429,0.500000,0.571429,0.571429,1,1
1,Is the agreement governed by US law?,[This agreement shall be governed by and const...,"Yes, governed by California (US) law",The agreement is governed by the laws of the S...,0.000000,0.149254,0.061538,0.149254,0.149254,1,1
2,Who bears liability for software malfunction?,[The vendor shall not be liable for indirect o...,The vendor is not liable,The vendor does not bear liability for indirec...,0.000000,0.300000,0.111111,0.300000,0.300000,1,1
3,Can the contract be renewed automatically?,[The contract will automatically renew for suc...,"Yes, auto-renews unless terminated","Yes, the contract can be renewed automatically.",0.000000,0.166667,0.000000,0.166667,0.166667,1,1
4,What is the payment due date?,[Payment is due within 15 days from the invoic...,15 days from invoice date,The payment is due within 15 days from the inv...,0.000000,0.625000,0.428571,0.625000,0.625000,1,1


In [28]:
prompt_v1 = "Context:\n{ctx}\n\nQ: {q}\nA:"
prompt_v2 = "You are a helpful legal assistant. Context:\n{ctx}\nAnswer the question: {q}"

q = df.iloc[1]["question"]
ctx = "\n".join(df.iloc[1]["contexts"][:2])

print("🔹 Prompt Version 1:\n", prompt_v1.format(ctx=ctx, q=q))
print("\n🔸 Prompt Version 2:\n", prompt_v2.format(ctx=ctx, q=q))

🔹 Prompt Version 1:
 Context:
This agreement shall be governed by and construed in accordance with the laws of the State of California.

Q: Is the agreement governed by US law?
A:

🔸 Prompt Version 2:
 You are a helpful legal assistant. Context:
This agreement shall be governed by and construed in accordance with the laws of the State of California.
Answer the question: Is the agreement governed by US law?
